#### The CelesTrack Dataset is very clean already, but mistaked can be happened by the dataset operators
Such as : 
- Missing values
- Duplication in the satellites
- value inconsistencies, etc

## Performing preprocssing on the satellite data
- Handle missing values
- Remove duplicates
- Structure data


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Important thorughout this project
%matplotlib inline


In [2]:
df = pd.read_csv('../data/01_raw/gp.csv')

In [3]:
df.head()

,OBJECT_NAME,OBJECT_ID,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,EPHEMERIS_TYPE,CLASSIFICATION_TYPE,NORAD_CAT_ID,ELEMENT_SET_NO,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT
0,CALSPHERE 1,1964-063C,2025-12-04T15:36:43.212960,13.763517,0.002716,90.2215,67.2697,159.0699,309.0581,0,U,900,999,4466,0.001228,1.211000e-05,0.0
1,CALSPHERE 2,1964-063E,2025-12-04T14:12:30.284352,13.528817,0.002040,90.2363,71.2062,74.4339,353.0789,0,U,902,999,82995,0.000097,7.400000e-07,0.0
2,LCS 1,1965-034C,2025-12-04T11:40:34.858848,9.893095,0.001344,32.1426,279.3197,153.3358,206.7832,0,U,1361,999,19060,0.001319,1.800000e-07,0.0
3,TEMPSAT 1,1965-065E,2025-12-04T15:49:06.875904,13.335813,0.007139,89.9887,212.6762,75.2749,339.0456,0,U,1512,999,93412,0.000160,8.900000e-07,0.0
4,CALSPHERE 4A,1965-065H,2025-12-04T16:19:00.877728,13.362372,0.006822,89.9093,124.3111,292.9140,93.8021,0,U,1520,999,93673,0.000319,1.750000e-06,0.0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13551 entries, 0 to 13550
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   OBJECT_NAME          13551 non-null  object 
 1   OBJECT_ID            13551 non-null  object 
 2   EPOCH                13551 non-null  object 
 3   MEAN_MOTION          13551 non-null  float64
 4   ECCENTRICITY         13551 non-null  float64
 5   INCLINATION          13551 non-null  float64
 6   RA_OF_ASC_NODE       13551 non-null  float64
 7   ARG_OF_PERICENTER    13551 non-null  float64
 8   MEAN_ANOMALY         13551 non-null  float64
 9   EPHEMERIS_TYPE       13551 non-null  int64  
 10  CLASSIFICATION_TYPE  13551 non-null  object 
 11  NORAD_CAT_ID         13551 non-null  int64  
 12  ELEMENT_SET_NO       13551 non-null  int64  
 13  REV_AT_EPOCH         13551 non-null  int64  
 14  BSTAR                13551 non-null  float64
 15  MEAN_MOTION_DOT      13551 non-null 

----
### Check for duplicate satellite records

In [5]:
df[df.duplicated()]

,OBJECT_NAME,OBJECT_ID,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,EPHEMERIS_TYPE,CLASSIFICATION_TYPE,NORAD_CAT_ID,ELEMENT_SET_NO,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT


- This dataset is very effective, it does not have any duplicate records.
- But, when building the final pipeline, it is still necessary to handle possible duplicates
- Here is a simple method to reliablely remove duplicates

In [6]:
df = df.drop_duplicates()

-------
### Handling missing values

In [7]:
df[df.isnull()]

,OBJECT_NAME,OBJECT_ID,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,EPHEMERIS_TYPE,CLASSIFICATION_TYPE,NORAD_CAT_ID,ELEMENT_SET_NO,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13546,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13547,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13548,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13549,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


- By looking at the result of df.info(), there are no missing values acorss the entire dataset.
- But, df[df.isnull()] returning wierd result

  
- original dataset was "full" (had no null values),  df.isnull() mask was all False. 
- When you use this all-False mask for indexing, pandas replaces every single value with NaN, giving you a DataFrame full of NaNs.

In [8]:
# A better reliable way to check if atleast one row has misisng values or not

rows_with_null = df.isnull().any(axis=1)

In [9]:
rows_with_null

0        False
1        False
2        False
3        False
4        False
         ...  
13546    False
13547    False
13548    False
13549    False
13550    False
Length: 13551, dtype: bool

In [10]:
df[rows_with_null]

,OBJECT_NAME,OBJECT_ID,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,EPHEMERIS_TYPE,CLASSIFICATION_TYPE,NORAD_CAT_ID,ELEMENT_SET_NO,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT


#### As the dataset contains no missing values, still there is a need for handling them

In [11]:
df.head()

,OBJECT_NAME,OBJECT_ID,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,EPHEMERIS_TYPE,CLASSIFICATION_TYPE,NORAD_CAT_ID,ELEMENT_SET_NO,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT
0,CALSPHERE 1,1964-063C,2025-12-04T15:36:43.212960,13.763517,0.002716,90.2215,67.2697,159.0699,309.0581,0,U,900,999,4466,0.001228,1.211000e-05,0.0
1,CALSPHERE 2,1964-063E,2025-12-04T14:12:30.284352,13.528817,0.002040,90.2363,71.2062,74.4339,353.0789,0,U,902,999,82995,0.000097,7.400000e-07,0.0
2,LCS 1,1965-034C,2025-12-04T11:40:34.858848,9.893095,0.001344,32.1426,279.3197,153.3358,206.7832,0,U,1361,999,19060,0.001319,1.800000e-07,0.0
3,TEMPSAT 1,1965-065E,2025-12-04T15:49:06.875904,13.335813,0.007139,89.9887,212.6762,75.2749,339.0456,0,U,1512,999,93412,0.000160,8.900000e-07,0.0
4,CALSPHERE 4A,1965-065H,2025-12-04T16:19:00.877728,13.362372,0.006822,89.9093,124.3111,292.9140,93.8021,0,U,1520,999,93673,0.000319,1.750000e-06,0.0


#### We will select those features having 'logical' importance to be completlely NaNs free
Here are the following features and their methods to handle them

1. Object Name -> Fill missing names with 'Unknown'
2. Object ID -> Fill missing IDs with 'Unknown'
3. EPOCH ->  If Feature Deriving a new Feature like Days since launch -> Fill accorindgly, else drop the row
4. MEAN_MOTION,	ECCENTRICITY,	INCLINATION, RA_OF_ASC_NODE,	ARG_OF_PERICENTER,	MEAN_ANOMALY
-> mostly dropping the rows but can be derived from other features
5. NORAD_CAT_ID -> Fill with 'Unknown'
6. ELEMENT_SET_NO -> Fill with Mode
7. REV_AT_EPOCH,	BSTAR
-> Mostly imputation (mean / median)
8. MEAN_MOTION_DOT	-> Fill with (mean / median) or any other method (check carefully as it is imp feature)
9. MEAN_MOTION_DDOT -> Fill with mode (as most of them are 0)
10. BSTAR -> Handle carefully or just drop the rows

- Some other features that may not be used at all
1. EPHEMERIS_TYPE,	CLASSIFICATION_TYPE -> We will still use them by filling by 'mode' for missing values

-----

In [12]:
# Fill Missing object name
df['OBJECT_NAME'] = df['OBJECT_NAME'].fillna('Unknown')

In [13]:
# Fill Missing object ID 
df['OBJECT_ID']= df['OBJECT_ID'].fillna('Unknown')

In [14]:
# handle missing epoch -> For now, we will drop the missing rows, later perform feature engineering to fill it.
df = df.dropna(subset = ['EPOCH'])

In [15]:
# Fill Missing MEAN_MOTION, ECCENTRICITY, INCLINATION, RA_OF_ASC_NODE, ARG_OF_PERICENTER, MEAN_ANOMALY -> Drop the Nulls for now
# These are very important TLE paramters, filling them with an easay method is not recommended, as wrong value can lead to anomolous results
df = df.dropna(subset = ['MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION', 'RA_OF_ASC_NODE', 'ARG_OF_PERICENTER', 'MEAN_ANOMALY'])

In [16]:
# Fill missing NORAD_CAT_ID -> Fill with 'Unknown'
df['NORAD_CAT_ID']= df['NORAD_CAT_ID'].fillna('Unknown')

In [17]:
# Fill missing ELEMENT_SET_NO -> Fill with imputation : Mode
df['ELEMENT_SET_NO']= df['ELEMENT_SET_NO'].fillna(df['ELEMENT_SET_NO'].mode())

In [18]:
# Fill missing REV_AT_EPOCH	 -> This is also a very important feature, missing with mean or median probably is not recommended
# So, we will drop the rows right now, later we will see more secured method

df = df.dropna(subset = ['REV_AT_EPOCH'])

In [19]:
# Fill missing BSTAR -> Very important feature, so we will drop the rows to avoid anomolous filling

df = df.dropna(subset = ['BSTAR'])

In [20]:
# Fill missing MEAN_MOTION_DOT, MEAN_MOTION_DDOT -> Same like Rev at epoch and Bstar, drop the rows
df = df.dropna(subset = ['MEAN_MOTION_DOT', 'MEAN_MOTION_DDOT'])

In [21]:
# Fill missing EPHEMERIS_TYPE, CLASSIFICATION_TYPE -> Fill each one by imputation  : Mode
df['EPHEMERIS_TYPE'] = df['EPHEMERIS_TYPE'].fillna(df['EPHEMERIS_TYPE'].mode())

df['CLASSIFICATION_TYPE'] = df['CLASSIFICATION_TYPE'].fillna(df['CLASSIFICATION_TYPE'].mode())

---
#### Save the cleaned dataset

In [22]:
df.to_csv('../data/02_cleaned/satellites_cleaned.csv', index=False)

In [23]:
df.head()

,OBJECT_NAME,OBJECT_ID,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,EPHEMERIS_TYPE,CLASSIFICATION_TYPE,NORAD_CAT_ID,ELEMENT_SET_NO,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT
0,CALSPHERE 1,1964-063C,2025-12-04T15:36:43.212960,13.763517,0.002716,90.2215,67.2697,159.0699,309.0581,0,U,900,999,4466,0.001228,1.211000e-05,0.0
1,CALSPHERE 2,1964-063E,2025-12-04T14:12:30.284352,13.528817,0.002040,90.2363,71.2062,74.4339,353.0789,0,U,902,999,82995,0.000097,7.400000e-07,0.0
2,LCS 1,1965-034C,2025-12-04T11:40:34.858848,9.893095,0.001344,32.1426,279.3197,153.3358,206.7832,0,U,1361,999,19060,0.001319,1.800000e-07,0.0
3,TEMPSAT 1,1965-065E,2025-12-04T15:49:06.875904,13.335813,0.007139,89.9887,212.6762,75.2749,339.0456,0,U,1512,999,93412,0.000160,8.900000e-07,0.0
4,CALSPHERE 4A,1965-065H,2025-12-04T16:19:00.877728,13.362372,0.006822,89.9093,124.3111,292.9140,93.8021,0,U,1520,999,93673,0.000319,1.750000e-06,0.0


In [24]:
len(df)

13551